In [1]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv(r"C:\projects_endtoend\loan_approval_prediction\notebooks\data\loan_prediction.csv")

In [4]:
data.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [5]:
data.drop("Loan_ID",axis=1,inplace=True)

In [6]:
X = data.drop('Loan_Status',axis=1)

In [7]:
X

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban
...,...,...,...,...,...,...,...,...,...,...,...
609,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural
610,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural
611,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban
612,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban


In [8]:
y=data[['Loan_Status']]

In [9]:
y

,Loan_Status
0,Y
1,N
2,Y
3,Y
4,Y
...,...
609,Y
610,Y
611,Y
612,Y


In [10]:
categorical = X.select_dtypes(include='object').columns.to_list()  ## categorical

In [11]:
categorical

['Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'Property_Area']

In [12]:
numerical = X.select_dtypes(exclude='object').columns.to_list()    ## numerical

In [13]:
numerical

['ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History']

In [14]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 614 entries, 0 to 613
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             601 non-null    object 
 1   Married            611 non-null    object 
 2   Dependents         599 non-null    object 
 3   Education          614 non-null    object 
 4   Self_Employed      582 non-null    object 
 5   ApplicantIncome    614 non-null    int64  
 6   CoapplicantIncome  614 non-null    float64
 7   LoanAmount         592 non-null    float64
 8   Loan_Amount_Term   600 non-null    float64
 9   Credit_History     564 non-null    float64
 10  Property_Area      614 non-null    object 
 11  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(7)
memory usage: 62.4+ KB


In [15]:
## numeric cols with mode imputation

mode_cols =  ['Credit_History','Loan_Amount_Term']

## Pipeline

In [16]:
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler  
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,FunctionTransformer  

from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer  

In [17]:
gender_cat = ['Male' 'Female']
married_cat = ['No' 'Yes']
depend_cat = ['0' '1' '2' '3+']
educ_cat = ['Graduate' 'Not Graduate']
self_cat = ['No' 'Yes']
property_cat = ['Urban' 'Rural' 'Semiurban']

In [18]:
# Use FunctionTransformer with np.log1p
log_transformer = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

In [19]:
# Define columns
numerical = ['ApplicantIncome', 'CoapplicantIncome']
mode_cols = ['Credit_History','Loan_Amount_Term','LoanAmount']
categorical = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']


In [20]:
num_pipeline = Pipeline(

    steps=[

        ("imputer",SimpleImputer(strategy='median')),  ## mean or median
        ("scaling",StandardScaler()),
        ("log",log_transformer)
        
    ]
)

In [21]:
mode_pipeline = Pipeline(

    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent'))
    ]
)

In [22]:
cat_pipeline = Pipeline(

    steps=[
        ("imputer",SimpleImputer(strategy='most_frequent')),
        ("encoding",OneHotEncoder(handle_unknown='ignore',categories=[gender_cat,married_cat,depend_cat,educ_cat,self_cat,property_cat]))
    ]
)

In [23]:
preprocessor = ColumnTransformer(

    [
        ("num_pipeline",num_pipeline,numerical), ## perform num_pipeline on num cols
        ("mode",mode_pipeline,mode_cols), ## perform mode_pipeline on mode cols
        ("cat_pipeline",cat_pipeline,categorical)  ## perform cat_pipeline on cat cols
    ]
)

In [24]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.30,random_state=30)

In [25]:
preprocessor.fit_transform(X_train)

array([[-1.0480398 ,  2.41852064,  1.        , ...,  0.        ,
         0.        ,  0.        ],
       [-1.14143355,  0.54140101,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.68783782, -0.04147198,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.05420688,  0.13499433,  1.        , ...,  0.        ,
         0.        ,  0.        ],
       [-2.04803202,  0.50239329,  1.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.66333614, -0.7277521 ,  0.        , ...,  0.        ,
         0.        ,  0.        ]])

In [26]:
preprocessor.get_feature_names_out()

array(['num_pipeline__ApplicantIncome', 'num_pipeline__CoapplicantIncome',
       'mode__Credit_History', 'mode__Loan_Amount_Term',
       'mode__LoanAmount', 'cat_pipeline__Gender_MaleFemale',
       'cat_pipeline__Married_NoYes', 'cat_pipeline__Dependents_0123+',
       'cat_pipeline__Education_GraduateNot Graduate',
       'cat_pipeline__Self_Employed_NoYes',
       'cat_pipeline__Property_Area_UrbanRuralSemiurban'], dtype=object)

In [27]:
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(preprocessor.transform(X_test),columns=preprocessor.get_feature_names_out())

In [28]:
X_train

,num_pipeline__ApplicantIncome,num_pipeline__CoapplicantIncome,mode__Credit_History,mode__Loan_Amount_Term,mode__LoanAmount,cat_pipeline__Gender_MaleFemale,cat_pipeline__Married_NoYes,cat_pipeline__Dependents_0123+,cat_pipeline__Education_GraduateNot Graduate,cat_pipeline__Self_Employed_NoYes,cat_pipeline__Property_Area_UrbanRuralSemiurban
0,-1.048040,2.418521,1.0,360.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.141434,0.541401,0.0,360.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.687838,-0.041472,0.0,180.0,113.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.322039,-0.727752,1.0,360.0,111.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.440526,-0.727752,0.0,300.0,152.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
424,0.478603,-0.727752,1.0,360.0,150.0,0.0,0.0,0.0,0.0,0.0,0.0
425,-0.815193,0.645896,1.0,360.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
426,-0.054207,0.134994,1.0,360.0,185.0,0.0,0.0,0.0,0.0,0.0,0.0
427,-2.048032,0.502393,1.0,480.0,113.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
X_test

,num_pipeline__ApplicantIncome,num_pipeline__CoapplicantIncome,mode__Credit_History,mode__Loan_Amount_Term,mode__LoanAmount,cat_pipeline__Gender_MaleFemale,cat_pipeline__Married_NoYes,cat_pipeline__Dependents_0123+,cat_pipeline__Education_GraduateNot Graduate,cat_pipeline__Self_Employed_NoYes,cat_pipeline__Property_Area_UrbanRuralSemiurban
0,-0.318962,-0.727752,1.0,360.0,76.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.609115,0.063100,1.0,180.0,182.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.511777,-0.727752,1.0,360.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.355727,0.344965,1.0,360.0,151.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.088196,-0.727752,1.0,360.0,144.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
180,-0.347001,-0.727752,1.0,360.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
181,-0.848107,0.121023,1.0,360.0,110.0,0.0,0.0,0.0,0.0,0.0,0.0
182,0.444472,0.397968,1.0,360.0,165.0,0.0,0.0,0.0,0.0,0.0,0.0
183,-0.494541,0.302227,1.0,360.0,110.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
X_train.isnull().sum()

num_pipeline__ApplicantIncome                      0
num_pipeline__CoapplicantIncome                    0
mode__Credit_History                               0
mode__Loan_Amount_Term                             0
mode__LoanAmount                                   0
cat_pipeline__Gender_MaleFemale                    0
cat_pipeline__Married_NoYes                        0
cat_pipeline__Dependents_0123+                     0
cat_pipeline__Education_GraduateNot Graduate       0
cat_pipeline__Self_Employed_NoYes                  0
cat_pipeline__Property_Area_UrbanRuralSemiurban    0
dtype: int64

# Model Training

In [31]:
## model training

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.model_selection import cross_val_score


In [32]:
import warnings
warnings.filterwarnings("ignore")

In [42]:
models = {

    'Logistic Regression': LogisticRegression(),
    'Decision Tree Classifier': DecisionTreeClassifier(),
    'Random Forest Classifier': RandomForestClassifier(),
    'Ada Boost Classifier': AdaBoostClassifier(),
    'KNeighbors Classifier': KNeighborsClassifier(),
    'SVC': SVC()
}

In [43]:
import numpy as np
def evaluate_model(true, predicted):
    accuracy  = accuracy_score(true, predicted)
    confusion = confusion_matrix(true, predicted)
    
    report = classification_report(true, predicted)
    return accuracy,confusion,report

In [35]:
model_list = []
accuracy_scores = []
confusion_matrices = []
report = []

In [36]:
for i in range(len(list(models))):
    model=list(models.values())[i]
    print(model)

LogisticRegression()
DecisionTreeClassifier()
RandomForestClassifier()
AdaBoostClassifier()
KNeighborsClassifier()
SVC()


In [45]:
for i in range(len(list(models))):
    model=list(models.values())[i]
    
    model.fit(X_train,y_train)

    #Make Predictions
    y_pred=model.predict(X_test)

    #this is a validation(test) score
    accuracy,confusion, report=evaluate_model(y_test,y_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model Training Performance')
    print("accuracy score:",accuracy)
    print("confusion matrix:",confusion)
    print("classification report",report)

    accuracy_scores.append(accuracy)
    confusion_matrices.append(confusion)
    
    print('='*100)
    print('\n')



Logistic Regression
Model Training Performance
accuracy score: 0.8
confusion matrix: [[ 18  33]
 [  4 130]]
classification report               precision    recall  f1-score   support

           N       0.82      0.35      0.49        51
           Y       0.80      0.97      0.88       134

    accuracy                           0.80       185
   macro avg       0.81      0.66      0.68       185
weighted avg       0.80      0.80      0.77       185



Decision Tree Classifier
Model Training Performance
accuracy score: 0.7513513513513513
confusion matrix: [[ 28  23]
 [ 23 111]]
classification report               precision    recall  f1-score   support

           N       0.55      0.55      0.55        51
           Y       0.83      0.83      0.83       134

    accuracy                           0.75       185
   macro avg       0.69      0.69      0.69       185
weighted avg       0.75      0.75      0.75       185



Random Forest Classifier
Model Training Performance
accuracy s